In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

from typing import TypedDict
from dotenv import load_dotenv
import os

In [12]:
load_dotenv()

True

In [13]:
# Read variables
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_API_BASE = os.getenv("OPENAI_API_BASE")
# MODEL_NAME = os.getenv("MODEL_NAME")

In [14]:
# Store them in environment variables
load_dotenv()
LM_STUDIO_BASE_URL = os.environ.get('LM_STUDIO_BASE_URL')
os.environ['LM_STUDIO_BASE_URL'] = LM_STUDIO_BASE_URL


In [16]:
# Create LLM
llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    openai_api_base=OPENAI_API_BASE,
    model="smallthinker-3b",
    temperature=0.7
)

print(llm.invoke("Explain RAG in simple words"))


content='So I\'ve been hearing about this term "RAG" lately, especially in the context of AI and how models like GPT work. But honestly, I\'m not entirely sure what it stands for or exactly what it means. I think I need to look into that.\n\nFirst off, I know that GPT is a type of language model developed by OpenAI. It\'s capable of generating human-like text based on the input it receives. So when people talk about RAG in relation to GPT, I imagine they\'re referring to how these models are being used or enhanced.\n\nI recall reading somewhere that "RAG" stands for "Relevant Answer Generator." That seems plausible because if a model is capable of generating answers to questions, it\'s probably doing so by finding relevant information within its training data. So, RAG could be about the process or component within these models that helps them generate appropriate answers based on the query.\n\nBut I\'m not entirely confident about that. Maybe I should check what OpenAI or other sources

In [17]:
print(llm)

output_version=None client=<openai.resources.chat.completions.completions.Completions object at 0x00000256659DAAA0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000256659DBD90> root_client=<openai.OpenAI object at 0x00000256659D8F70> root_async_client=<openai.AsyncOpenAI object at 0x00000256659D9CF0> model_name='smallthinker-3b' temperature=0.7 model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='http://192.168.56.1:1234/v1' openai_proxy=None


In [18]:
# create a state 

class LLMState(TypedDict): 

    question: str
    answer: str 

In [20]:
def llm_qa(state : LLMState) -> LLMState:
    
    # extract the question from the state
    question = state["question"]


    # form a prompt 
    prompt = f"Answer the following question : {question}"



    # ask the question to the llm 
    answer = llm.invoke(prompt).content


    # update the answer in the state 
    state['answer'] = answer 

    return state 

In [23]:
# create our graph 

graph = StateGraph(LLMState)


# add nodes 

graph.add_node("llm_qa",llm_qa)


# add edges 

graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa",END)


# compile the graph 
workflow = graph.compile()

In [26]:
# execute the graph 
intial_state = {"question": "what far is moon from the earth ?"}


workflow.invoke(intial_state)


{'question': 'what far is moon from the earth ?',
 'answer': 'So I need to figure out how far away the Moon is from Earth. I remember that the distance between Earth and the Moon varies because both the Earth and the Moon are moving in their orbits around the Sun, but overall, the average distance should be fairly constant.\n\nFirst, I think about what unit of measurement is commonly used for such vast distances. Kilometers or miles come to mind, but I\'m not sure which one would be more appropriate here. Maybe meters? Wait, no, even that might be too small. Let\'s see, Earth\'s diameter is about 12,742 kilometers, and the Moon\'s average distance from Earth is roughly 384,400 kilometers or miles.\n\nWait, miles are used in the United States, while kilometers are international. Maybe I should think in terms of astronomical units (AU), where 1 AU is approximately the average distance from Earth to Sun, which is about 150 million kilometers. So, the Moon\'s average distance would be abou